In [1]:
# Install all required packages
import sys
!{sys.executable} -m pip install sentence-transformers faiss-cpu PyPDF2 streamlit numpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached faiss_cpu-1.13.0-cp313-cp313-win_amd64.whl.metadata (7.7 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.9.1-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.0 MB 2.6 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/12.0 M

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 [torch]
   ----------- ---------------------------- 2/7 

In [2]:
import PyPDF2
import re
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [3]:
def extract_text_from_pdf(pdf_path):
    """Extract text from PDF file"""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            print(f"📄 Total pages: {len(pdf_reader.pages)}")
            
            for page_num, page in enumerate(pdf_reader.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + " "
                    
            print(f"✅ Extracted {len(text)} characters")
        return text
    except Exception as e:
        print(f"❌ Error: {e}")
        return ""

# Read your PDF
pdf_path = "student_handbook.pdf"
handbook_text = extract_text_from_pdf(pdf_path)

# Show first 500 characters
print("\n📖 Sample text from handbook:")
print(handbook_text[:500])

📄 Total pages: 97
✅ Extracted 100013 characters

📖 Sample text from handbook:
CHRISLAND UNIVERSITY
INTELLECTUAL  RADIANCESTUDENT 
HANDBOOKABEOKUTA, OGUN STATE, NIGERIA i 
  
 
 
 
 
CHRISLAND UNIVERSITY  
...intellectual  radiance  
 
 
 
 
 
STUDENT HANDBOOK  
 
 
 
 ii 
  
Nigerian National Anthem  
(I) Nigeria, we hail thee,  
Our own dear native land,  
Though tribes and tongues may differ  
In brotherhood we stand,  
Nigerians all, and proud to serve  
Our sovereign Motherland.  
(II) Our flag shall be a symbol  
That truth and justice reign,  
In peace or battle hon


In [4]:
def split_into_chunks(text, chunk_size=400):
    """Break text into smaller pieces"""
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text.strip())
    
    # Split by sentences
    sentences = re.split(r'[.!?]+', text)
    
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    # Remove very short chunks
    chunks = [chunk for chunk in chunks if len(chunk) > 50]
    
    return chunks

# Split the text
text_chunks = split_into_chunks(handbook_text, chunk_size=400)

print(f"✅ Created {len(text_chunks)} chunks")
print(f"\n📝 Sample chunk 1:")
print(text_chunks[0])
print(f"\n📝 Sample chunk 2:")
print(text_chunks[1])

✅ Created 287 chunks

📝 Sample chunk 1:
CHRISLAND UNIVERSITY INTELLECTUAL RADIANCESTUDENT HANDBOOKABEOKUTA, OGUN STATE, NIGERIA i CHRISLAND UNIVERSITY. intellectual radiance STUDENT HANDBOOK ii Nigerian National Anthem (I) Nigeria, we hail thee, Our own dear native land, Though tribes and tongues may differ In brotherhood we stand, Nigerians all, and proud to serve Our sovereign Motherland.

📝 Sample chunk 2:
(II) Our flag shall be a symbol That truth and justice reign, In peace or battle honour'd, And this we count as gain, To hand on to our children A banner without stain. (III) O God of all creation, Grant this our one request. Help us to build a nation Where no man is oppressed, And so with peace and plenty Nigeria may be blessed.


In [5]:
print("🔄 Loading AI model... (This may take 1-2 minutes on first run)")

# Load the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Model loaded successfully!")
print(f"Model dimension: {model.get_sentence_embedding_dimension()}")

🔄 Loading AI model... (This may take 1-2 minutes on first run)
✅ Model loaded successfully!
Model dimension: 384


In [6]:
print("🔄 Creating embeddings for all chunks...")
print("This converts text into numbers that AI can search through")

# Create embeddings
embeddings = model.encode(text_chunks, show_progress_bar=True)

print(f"\n✅ Created embeddings!")
print(f"Shape: {embeddings.shape}")
print(f"Each chunk is now represented by {embeddings.shape[1]} numbers")

# Show sample embedding
print(f"\n🔢 Sample embedding (first 10 numbers):")
print(embeddings[0][:10])

🔄 Creating embeddings for all chunks...
This converts text into numbers that AI can search through


Batches:   0%|          | 0/9 [00:00<?, ?it/s]


✅ Created embeddings!
Shape: (287, 384)
Each chunk is now represented by 384 numbers

🔢 Sample embedding (first 10 numbers):
[ 0.00846716  0.06573722 -0.04724444 -0.09826555  0.11039788 -0.04913084
 -0.00691152 -0.05026208  0.0067364   0.02095895]


In [7]:
print("🔄 Building FAISS search index...")

# Get dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add all embeddings to the index
index.add(embeddings.astype('float32'))

print(f"✅ FAISS index created!")
print(f"Total vectors in index: {index.ntotal}")
print(f"Dimension: {dimension}")

🔄 Building FAISS search index...
✅ FAISS index created!
Total vectors in index: 287
Dimension: 384


In [8]:
def search_handbook(query, top_k=3):
    """Search for relevant information in handbook"""
    print(f"🔍 Searching for: '{query}'")
    
    # Convert query to embedding
    query_embedding = model.encode([query])
    
    # Search in FAISS index
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    # Get relevant chunks
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'rank': i + 1,
            'chunk': text_chunks[idx],
            'distance': distances[0][i]
        })
    
    return results

print("✅ Search function ready!")

✅ Search function ready!


In [9]:
# Test with a question
question = "What are the admission requirements?"

results = search_handbook(question, top_k=3)

print(f"\n❓ Question: {question}")
print("="*80)

for result in results:
    print(f"\n🎯 Result #{result['rank']} (Distance: {result['distance']:.2f})")
    print("-"*80)
    print(result['chunk'][:400] + "...")
    print()

🔍 Searching for: 'What are the admission requirements?'

❓ Question: What are the admission requirements?

🎯 Result #1 (Distance: 1.02)
--------------------------------------------------------------------------------
Students election shall be governed by SRC Election Guidelines as may be issued by the relevant university authority from time to time. Eligibility /Nominations: i. To be eligible to contest for any position such as President, Secretary, Sports Coordinator, Welfare Secretary, Hall representative, or Social Coordinator , such c andidates must be in 300 level and above and must have a minimum of 3....


🎯 Result #2 (Distance: 1.05)
--------------------------------------------------------------------------------
All fresh students must be cleared in sequence by the Admissions Office, Bursary and Student Affairs Unit on their first arrival. b. All students must reg ister at the beginning of each Semester for the courses prescribed by their colleges. c. Registration must be com

In [10]:
def ask_question(question):
    """Ask a question and get formatted answer"""
    results = search_handbook(question, top_k=2)
    
    print(f"\n{'='*80}")
    print(f"❓ YOUR QUESTION: {question}")
    print(f"{'='*80}\n")
    
    print("🤖 ANSWER FROM HANDBOOK:\n")
    
    for i, result in enumerate(results, 1):
        print(f"📌 Section {i}:")
        print(result['chunk'][:350])
        print("\n")
    
    print("="*80)

# Try different questions
print("🎯 Testing different questions:\n")

# Question 1
ask_question("What are the admission requirements?")

# Question 2
ask_question("Tell me about the grading system")

# Question 3
ask_question("What are the academic policies?")

🎯 Testing different questions:

🔍 Searching for: 'What are the admission requirements?'

❓ YOUR QUESTION: What are the admission requirements?

🤖 ANSWER FROM HANDBOOK:

📌 Section 1:
Students election shall be governed by SRC Election Guidelines as may be issued by the relevant university authority from time to time. Eligibility /Nominations: i. To be eligible to contest for any position such as President, Secretary, Sports Coordinator, Welfare Secretary, Hall representative, or Social Coordinator , such c andidates must be in 


📌 Section 2:
All fresh students must be cleared in sequence by the Admissions Office, Bursary and Student Affairs Unit on their first arrival. b. All students must reg ister at the beginning of each Semester for the courses prescribed by their colleges. c. Registration must be completed within the prescribed period.


🔍 Searching for: 'Tell me about the grading system'

❓ YOUR QUESTION: Tell me about the grading system

🤖 ANSWER FROM HANDBOOK:

📌 Section 1:
The

In [ ]:
# Now you can test with your own questions!

def interactive_chat():
    """Interactive Q&A session"""
    print("💬 Chrisland University Assistant")
    print("="*50)
    print("Type 'quit' to exit\n")
    
    while True:
        question = input("❓ Your question: ")
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("👋 Goodbye!")
            break
        
        if question.strip():
            ask_question(question)

# Run interactive chat
interactive_chat()

💬 Chrisland University Assistant
Type 'quit' to exit



❓ Your question:  what is the grade for 30 marks


🔍 Searching for: 'what is the grade for 30 marks'

❓ YOUR QUESTION: what is the grade for 30 marks

🤖 ANSWER FROM HANDBOOK:

📌 Section 1:
54 CHAPTER TWENTY RULES AND REGULATIONS GOVERNING THE CONDUCT OF EXAMINATIONS ELIGIBILITY All students who are registered for courses in a given Semester are eligible to sit for examinations in those courses except students in the following categories: a.


📌 Section 2:
…………………… Signature of Student/Date Signature(s) of Parent(s)/Date NOTE: The rules and regulations are subject to review by the University Management /Senate, therefore , students are 80 expected to strictly abide by rules and regulations that may be prescribed from time to time.


